In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

CATALOG = "sap_lakehouse"


BRONZE_EVENTS_TABLE = f"{CATALOG}.bronze.invoice_status_events"
SILVER_INVOICES_TABLE = f"{CATALOG}.silver.current_invoice_status"
GOLD_ANALYTICS_TABLE = f"{CATALOG}.gold.invoice_status_summary"
AUDIT_TABLE = f"{CATALOG}.audit.event_pipeline_runs"
QUARANTINE_TABLE = f"{CATALOG}.quarantine.bad_invoice_events"


EVENT_CONTRACT_PATH = "/Volumes/sap_lakehouse/bronze/sap_api_contracts/InvoiceStatusChangeSpec.json"

In [0]:
contract_df = spark.read.option("multiple", "true").json(EVENT_CONTRACT_PATH)
display(contract_df)

In [0]:
sap_invoice_events = [

    {
        "event_id": "EVT001",
        "invoice_id": "INV001",
        "vendor_id": "V001",
        "company_code": "1000",
        "status": "SUBMITTED",
        "amount": 5000.00,
        "currency": "USD",
        "event_timestamp": "2026-05-01T08:00:00"
    },

    {
        "event_id": "EVT002",
        "invoice_id": "INV001",
        "vendor_id": "V001",
        "company_code": "1000",
        "status": "APPROVED",
        "amount": 5000.00,
        "currency": "USD",
        "event_timestamp": "2026-05-01T10:00:00"
    },

    {
        "event_id": "EVT003",
        "invoice_id": "INV002",
        "vendor_id": "V002",
        "company_code": "1000",
        "status": "SUBMITTED",
        "amount": 3200.00,
        "currency": "USD",
        "event_timestamp": "2026-05-01T09:00:00"
    },

    {
        "event_id": "EVT004",
        "invoice_id": "INV002",
        "vendor_id": "V002",
        "company_code": "1000",
        "status": "REJECTED",
        "amount": 3200.00,
        "currency": "USD",
        "event_timestamp": "2026-05-01T11:00:00"
    },

    {
        "event_id": "EVT005",
        "invoice_id": "INV003",
        "vendor_id": None,
        "company_code": "2000",
        "status": "SUBMITTED",
        "amount": 900.00,
        "currency": "USD",
        "event_timestamp": "2026-05-02T13:00:00"
    }

]

In [0]:
events_df = (
    spark.createDataFrame(sap_invoice_events)
    .withColumn(
        "event_timestamp",
        F.to_timestamp("event_timestamp")
    )
    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )
)

display(events_df)

In [0]:
events_df.write.format("delta") \
    .mode("append") \
    .saveAsTable(BRONZE_EVENTS_TABLE)

In [0]:
display(
    spark.table(BRONZE_EVENTS_TABLE)
)

In [0]:
bronze_df = spark.table(BRONZE_EVENTS_TABLE)

valid_events_df = bronze_df.filter(
    F.col("event_id").isNotNull() &
    F.col("invoice_id").isNotNull() &
    F.col("vendor_id").isNotNull() &
    F.col("company_code").isNotNull() &
    F.col("status").isNotNull() &
    F.col("event_timestamp").isNotNull()
)

bad_events_df = bronze_df.subtract(valid_events_df)

if bad_events_df.count() > 0:

    bad_events_df.write.format("delta") \
        .mode("append") \
        .saveAsTable(QUARANTINE_TABLE)

display(bad_events_df)

In [0]:
event_window = Window.partitionBy("event_id") \
    .orderBy(F.col("ingestion_timestamp").desc())

deduped_events_df = (
    valid_events_df
    .withColumn("row_num", F.row_number().over(event_window))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

display(deduped_events_df)

In [0]:
invoice_window = Window.partitionBy("invoice_id") \
    .orderBy(
        F.col("event_timestamp").desc(),
        F.col("ingestion_timestamp").desc()
    )

latest_invoice_state_df = (
    deduped_events_df
    .withColumn("row_num", F.row_number().over(invoice_window))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
    .withColumn("last_processed_at", F.current_timestamp())
)

display(latest_invoice_state_df)

In [0]:
if spark.catalog.tableExists(SILVER_INVOICES_TABLE):

    silver_table = DeltaTable.forName(spark, SILVER_INVOICES_TABLE)

    silver_table.alias("target").merge(
        latest_invoice_state_df.alias("source"),
        "target.invoice_id = source.invoice_id"
    ).whenMatchedUpdate(
        condition="source.event_timestamp >= target.event_timestamp",
        set={
            "event_id": "source.event_id",
            "vendor_id": "source.vendor_id",
            "company_code": "source.company_code",
            "status": "source.status",
            "amount": "source.amount",
            "currency": "source.currency",
            "event_timestamp": "source.event_timestamp",
            "ingestion_timestamp": "source.ingestion_timestamp",
            "last_processed_at": "source.last_processed_at"
        }
    ).whenNotMatchedInsertAll().execute()

else:
    latest_invoice_state_df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(SILVER_INVOICES_TABLE)

In [0]:
display(
    spark.table(SILVER_INVOICES_TABLE)
    .orderBy("invoice_id")
)